# Ingest: TPC-DS Retail Data Warehouse

Retail & supply-chain demo data. Unlike the other generators in this
repo, there's no external file to download here — this notebook
generates the TPC-DS benchmark dataset directly, using DuckDB's bundled
`tpcds` extension (which implements the TPC's own official `dsdgen`
reference generator). TPC-DS is the Transaction Processing Performance
Council's industry-standard benchmark for retail/supply-chain data
warehouses — the schema and generation algorithm are themselves the
real, well-established artifact, not a convenience-picked file, so this
still fits the "real, industry data" bar the other six verticals use,
just deterministically generated rather than downloaded.

Persists a representative 12-table subset of the full 24-table TPC-DS
schema at scale factor 1 (`sf=1`, ~1GB nominal across the full schema):
all three sales channels (`store_sales`, `catalog_sales`, `web_sales`)
plus the dimensions needed to build a cross-channel revenue story
(`date_dim`, `item`, `promotion`, `store`, `warehouse`), customer breadth
(`customer`, `customer_address`, `customer_demographics`), and
`inventory`.

Approximate row counts at `sf=1`: `store_sales` ~2.88M, `catalog_sales`
~1.44M, `web_sales` ~0.72M, `inventory` ~11.75M — the largest table
here. Several dimension tables (`customer_demographics` ~1.92M,
`date_dim` ~73K) are fixed-size by spec regardless of scale factor. All
12 tables keep their standard TPC-DS table/column names verbatim.

No download or auth required — `dsdgen` generates deterministically, so
re-running reproduces the same data. Re-runnable: `CREATE DATABASE IF
NOT EXISTS` and `mode("overwrite")`, so re-running just refreshes the
tables.

In [ ]:
%pip install -q duckdb pandas pyarrow

import duckdb
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

## Configuration

In [ ]:
DATABASE_NAME = "commerce"
SCALE_FACTOR = 1

TABLES = [
    "store_sales",
    "catalog_sales",
    "web_sales",
    "inventory",
    "date_dim",
    "item",
    "promotion",
    "store",
    "warehouse",
    "customer",
    "customer_address",
    "customer_demographics",
]

## Generate TPC-DS benchmark data

`dsdgen` populates DuckDB's own catalog with the full 24-table TPC-DS
schema; only the tables listed in `TABLES` above get pulled into Spark
below.

In [ ]:
import os

EXPORT_DIR = "/tmp/tpcds_export"
os.makedirs(EXPORT_DIR, exist_ok=True)

con = duckdb.connect(f"{EXPORT_DIR}/tpcds.duckdb")
con.sql("INSTALL tpcds; LOAD tpcds;")
con.sql(f"CALL dsdgen(sf={SCALE_FACTOR});")


## Create schema/database

In [ ]:
spark.sql(f"""
CREATE DATABASE IF NOT EXISTS {DATABASE_NAME}
COMMENT 'TPC-DS industry-standard retail data warehouse benchmark (sf={SCALE_FACTOR}), used for omnichannel retail demonstrations'
""")

## Persist as managed Spark tables

In [ ]:
for table_name in TABLES:
    parquet_path = f"{EXPORT_DIR}/{table_name}.parquet"
    con.sql(f"COPY {table_name} TO '{parquet_path}' (FORMAT PARQUET)")
    sdf = spark.read.parquet(parquet_path)
    sdf.write \
        .mode("overwrite") \
        .saveAsTable(f"{DATABASE_NAME}.{table_name}")
    os.remove(parquet_path)

## Verify

In [ ]:
for table_name in TABLES:
    count = spark.table(f"{DATABASE_NAME}.{table_name}").count()
    print(f"{DATABASE_NAME}.{table_name}: {count:,} rows")